In [2]:
# ============================================================
# Multimodal Fusion: Digital (ResNet50) + Thermal (DenseNet121)
# ============================================================
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [3]:
# ---------------------- CONFIG ----------------------
digital_model_path = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\digital_resnet50_final.h5"
thermal_model_path = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\densenet121_thermal_guava_final.h5"

digital_data_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\train"
thermal_data_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\train"
val_digital_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\val"
val_thermal_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\val"

In [4]:
BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)
EPOCHS = 50
LR = 1e-4
NUM_CLASSES = 3

In [5]:
# ============================================================
# STEP 1: Load pretrained models
# ============================================================
print("Loading pretrained models...")
digital_base = load_model(digital_model_path)
thermal_base = load_model(thermal_model_path)

# Extract feature layers (remove classifier heads)
digital_feat = Model(inputs=digital_base.input, outputs=digital_base.layers[-2].output)
thermal_feat = Model(inputs=thermal_base.input, outputs=thermal_base.layers[-2].output)

# Freeze feature extractors
digital_feat.trainable = False
thermal_feat.trainable = False

Loading pretrained models...


In [6]:
# ============================================================
# STEP 2: Data Generators
# ============================================================
print("Creating data generators...")

digital_datagen = ImageDataGenerator(rescale=1.0 / 255)
thermal_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Training generators (shuffle=False ensures order alignment)
digital_train_gen = digital_datagen.flow_from_directory(
    digital_data_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

thermal_train_gen = thermal_datagen.flow_from_directory(
    thermal_data_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Validation generators
digital_val_gen = digital_datagen.flow_from_directory(
    val_digital_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

thermal_val_gen = thermal_datagen.flow_from_directory(
    val_thermal_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Creating data generators...
Found 6531 images belonging to 3 classes.
Found 6639 images belonging to 3 classes.
Found 1398 images belonging to 3 classes.
Found 1421 images belonging to 3 classes.


In [7]:
# ============================================================
# STEP 3: Fusion Generator (Pairs both inputs)
# ============================================================
def fusion_generator(gen1, gen2):
    """Yield paired (digital, thermal) batches."""
    while True:
        X1, y1 = next(gen1)
        X2, y2 = next(gen2)
        # Ensure same batch size alignment
        b = min(len(X1), len(X2))
        yield [X1[:b], X2[:b]], y1[:b]

train_generator = fusion_generator(digital_train_gen, thermal_train_gen)
val_generator = fusion_generator(digital_val_gen, thermal_val_gen)

steps_per_epoch = min(len(digital_train_gen), len(thermal_train_gen))
val_steps = min(len(digital_val_gen), len(thermal_val_gen))

In [8]:
# ============================================================
# STEP 4: Fusion Model
# ============================================================
print("Building fusion model...")

input_digital = Input(shape=(224, 224, 3))
input_thermal = Input(shape=(224, 224, 3))

digital_features = digital_feat(input_digital)
thermal_features = thermal_feat(input_thermal)

# Optional projection to same dimension before concatenation
digital_features = Dense(512, activation='relu')(digital_features)
thermal_features = Dense(512, activation='relu')(thermal_features)

fusion = Concatenate()([digital_features, thermal_features])
fusion = Dense(512, activation='relu')(fusion)
fusion = Dropout(0.4)(fusion)
fusion = Dense(128, activation='relu')(fusion)
fusion = Dropout(0.3)(fusion)
output = Dense(NUM_CLASSES, activation='softmax')(fusion)

fusion_model = Model(inputs=[input_digital, input_thermal], outputs=output)

fusion_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fusion_model.summary()

Building fusion model...


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 2048)      │ 23,587,712 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 1024)      │  7,037,504 │ input_layer_1[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,049,088 │ functional[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    524,800 │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    524,800 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     65,664 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 3)         │        387 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,789,955 (125.08 MB)

 Trainable params: 2,164,739 (8.26 MB)

 Non-trainable params: 30,625,216 (116.83 MB)

In [9]:
# ============================================================
# STEP 5: Callbacks
# ============================================================
callbacks = [
    ModelCheckpoint("best_fusion_model.h5", save_best_only=True, monitor='val_accuracy', mode='max'),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

In [10]:
# ============================================================
# STEP 6: Train Fusion Model (Fixed Version, 50 Epochs)
# ============================================================
print("\n🚀 Starting fusion model training...")

# --- Set Epochs ---
EPOCHS = 50  # ✅ Run for 50 epochs

# --- Wrapper Generator to Combine Both Modalities ---
def fusion_generator(digital_gen, thermal_gen):
    while True:
        digital_batch = next(digital_gen)
        thermal_batch = next(thermal_gen)

        # Get minimum batch size to ensure alignment
        min_len = min(len(digital_batch[0]), len(thermal_batch[0]))

        # Ensure same size and tuple format for Keras
        digital_imgs = digital_batch[0][:min_len]
        thermal_imgs = thermal_batch[0][:min_len]
        labels = digital_batch[1][:min_len]  # assume both have same labels

        yield ((digital_imgs, thermal_imgs), labels)  # ✅ tuple, not list!

# --- Steps per Epoch ---
steps_per_epoch = min(len(digital_train_gen), len(thermal_train_gen))
val_steps = min(len(digital_val_gen), len(thermal_val_gen))

# --- Train Fusion Model ---
history = fusion_model.fit(
    fusion_generator(digital_train_gen, thermal_train_gen),
    validation_data=fusion_generator(digital_val_gen, thermal_val_gen),
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("✅ Fusion model training complete after 50 epochs!")


🚀 Starting fusion model training...
Epoch 1/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9354 - loss: 0.2884

409/409 ━━━━━━━━━━━━━━━━━━━━ 830s 2s/step - accuracy: 0.9464 - loss: 0.3430 - val_accuracy: 0.3054 - val_loss: 6.3715 - learning_rate: 1.0000e-04
Epoch 2/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 751s 2s/step - accuracy: 0.8665 - loss: 0.6258 - val_accuracy: 0.3054 - val_loss: 5.6355 - learning_rate: 1.0000e-04
Epoch 3/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 749s 2s/step - accuracy: 0.8498 - loss: 0.6646 - val_accuracy: 0.3054 - val_loss: 5.9383 - learning_rate: 1.0000e-04
Epoch 4/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6772 - loss: 1.4354

409/409 ━━━━━━━━━━━━━━━━━━━━ 753s 2s/step - accuracy: 0.8133 - loss: 0.7590 - val_accuracy: 0.3061 - val_loss: 6.3357 - learning_rate: 1.0000e-04
Epoch 5/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 739s 2s/step - accuracy: 0.8394 - loss: 0.7028 - val_accuracy: 0.3061 - val_loss: 5.6098 - learning_rate: 1.0000e-04
Epoch 6/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 725s 2s/step - accuracy: 0.8276 - loss: 0.7125 - val_accuracy: 0.3061 - val_loss: 5.7424 - learning_rate: 1.0000e-04
Epoch 7/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 697s 2s/step - accuracy: 0.8262 - loss: 0.7502 - val_accuracy: 0.3061 - val_loss: 5.5549 - learning_rate: 1.0000e-04
Epoch 8/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.8296 - loss: 0.6763 - val_accuracy: 0.3061 - val_loss: 5.7536 - learning_rate: 1.0000e-04
Epoch 9/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 677s 2s/step - accuracy: 0.8191 - loss: 0.7584 - val_accuracy: 0.3061 - val_loss: 6.8282 - learning_rate: 1.0000e-04
✅ Fusion model training complete after 50 epochs!
